In [39]:
def pp(msg=None, value=None):
    if msg is not None:
        print(msg)
    if value is not None:
        print(value)
    print('=' * 100)

1 The mean and the median
Load the table. First, make a small pd.Series with five numbers of reviews that are close to
each other, and calculate the mean and the median. Then add one listing with 1293
reviews (the most popular one) and calculate again. What happened to each number?
Then calculate the mean and the median of the real column number_of_reviews for all
4,574 listings. Which one would you use to answer "how many reviews does a typical
listing have?", and why?

Answer:
With close numbers mean and median are the same, with standout numbers - different.
As there are a lot of standout values, I would choose median.

In [40]:
import pandas as pd

listings_raw = pd.read_csv('listings_clean.csv', 
                           parse_dates=['last_scraped', 'calendar_last_scraped', 'first_review', 'last_review'])
listings = listings_raw.copy()
pp(listings.shape)
# pp(listings.dtypes)

five = pd.Series([100, 200, 300, 400, 500])
pp(five.mean())
pp(five.median())

six = pd.Series([100, 200, 300, 400, 500, 1293])
pp(six.mean())
pp(six.median())

# pp(listings['number_of_reviews'].value_counts())
pp(listings['number_of_reviews'].mean().round(1))
pp(listings['number_of_reviews'].median())


(4574, 73)
300.0
300.0
465.5
350.0
35.8
9.0


2 Percentiles and the spread
Run describe() on number_of_reviews. Then calculate the 10th, 25th, 50th, 75th, 90th and
99th percentile with quantile. Calculate q1, q3 and the iqr. Write one sentence in your own
words about what the IQR says here. Look at the 10th percentile: what does it tell you
about how many listings have no reviews at all?

Answer: 
IQR (Interquartile Range) tells about the reviews 50% of the listings have (27 reviews or less).
10th percentile tells that 10% of listings do not have any review.

In [41]:
pp(listings['number_of_reviews'].describe())

pp(listings['number_of_reviews'].quantile([0.1, 0.25, 0.5, 0.75, 0.9, 0.99]))

q1 = listings['number_of_reviews'].quantile(0.25)
q3 = listings['number_of_reviews'].quantile(0.75)
iqr = q3 - q1
pp(iqr)

count    4574.000000
mean       35.754045
std        78.377200
min         0.000000
25%         2.000000
50%         9.000000
75%        29.000000
max      1293.000000
Name: number_of_reviews, dtype: float64
0.10      0.00
0.25      2.00
0.50      9.00
0.75     29.00
0.90     94.00
0.99    400.27
Name: number_of_reviews, dtype: float64
27.0


3 The fences and the outliers
Calculate the upper and the lower fence with the IQR rule. Can there be outliers below the
lower fence for this column? Why, or why not? How many listings are above the upper
fence? Then look at them: how many of each room type, and what is their median price
compared with all the other listings? Write what you see in a comment.

Answer:
No, `lower` fence is negative, we cannot have negative number of review.
Median for outliers is much more bigger, because outliers review number influence `p50` a lot.

In [42]:
lower = q1 - 1.5 * iqr
upper = q3 + 1.5 * iqr
print(round(lower, 1), round(upper, 1))

outliers = listings[listings['number_of_reviews'] > upper]
print(len(outliers))

pp(outliers['room_type'].value_counts())

median_outliers = outliers['number_of_reviews'].median()
median_others = listings[listings['number_of_reviews'] <= upper]['number_of_reviews'].median()

pp("Median for outliers:", median_outliers)
pp("Median for others:", median_others)

-38.5 69.5
582
room_type
Entire home/apt    446
Private room       121
Shared room         14
Hotel room           1
Name: count, dtype: int64
Median for outliers:
148.5
Median for others:
7.0


4 Mark them, and compare
Do not remove the outliers. Mark them in a new column, reviews_outlier (True or False),
and count them. Then make a table with pd.DataFrame that shows count, mean, median and
std for number_of_reviews, with all listings and without the outliers. How many percent
does each number change when the outliers are gone? Which changes the most, and
which the least?

Answer:
most - std
least - median

In [43]:
listings['reviews_outlier'] = listings['number_of_reviews'] > upper
pp(listings['reviews_outlier'].value_counts())

compare = pd.DataFrame({
    'all': listings['number_of_reviews']
        .agg(['count', 'mean', 'median', 'std']),
    'without_outliers' : listings.loc[~listings['reviews_outlier'], 'number_of_reviews']
        .agg(['count', 'mean', 'median', 'std']),
}).round(0)
pp(compare)

diff = (compare['all'] - compare['without_outliers']) / compare['all'] * 100
pp("% difference:", diff.round(1))



reviews_outlier
False    3992
True      582
Name: count, dtype: int64
           all  without_outliers
count   4574.0            3992.0
mean      36.0              13.0
median     9.0               7.0
std       78.0              16.0
% difference:
count     12.7
mean      63.9
median    22.2
std       79.5
dtype: float64


5 Describe the groups
Use groupby and describe to get all the numbers for number_of_reviews for every room_type,
rounded to 0 decimals. Which room type has the highest median? Which group do you not
want to say anything about, and why? (Hint: look at the count column.)

Answer:
highest median - Shared room
not to say anything about - Hotel room (only 3 reviews)

In [44]:
pp(
    listings.groupby('room_type')['number_of_reviews'].describe().round(0)
)

                  count   mean    std   min   25%   50%    75%     max
room_type                                                             
Entire home/apt  3760.0   34.0   76.0   0.0   2.0   8.0   27.0  1293.0
Hotel room          3.0   61.0   16.0  49.0  52.0  56.0   68.0    79.0
Private room      787.0   41.0   84.0   0.0   2.0  12.0   38.0   867.0
Shared room        24.0  114.0  122.0   0.0  28.0  82.0  146.0   510.0


6 Correlation
Calculate the correlation between number_of_reviews and these columns: price,
accommodates, bedrooms, availability_365, review_scores_rating, minimum_nights and
days_since_review. Which is the strongest connection, which is the weakest, and which are
negative? Then calculate the correlation between number_of_reviews and availability_365
with and without the outliers from task 4. What happens, and what does it tell you about
the first number?

Answer:
Without outliers there is no connection between 'number_of_reviews' and 'availability_365'.

In [51]:
columns = [
    'number_of_reviews',
    'price',
    'accommodates', 
    'bedrooms', 
    'availability_365', 
    'review_scores_rating', 
    'minimum_nights', 
    'days_since_review'
]
pp(listings[columns].round(2).corr()['number_of_reviews'])

no_outliers = listings[~listings['reviews_outlier']]
print(listings['number_of_reviews'].corr(listings['availability_365']).round(2))
print(no_outliers['number_of_reviews'].corr(no_outliers['availability_365']).round(2))

number_of_reviews       1.000000
price                  -0.091789
accommodates           -0.050848
bedrooms               -0.144178
availability_365        0.183156
review_scores_rating   -0.039295
minimum_nights         -0.065609
days_since_review      -0.185301
Name: number_of_reviews, dtype: float64
0.18
0.0
